# Prophecy — Smart Reassort


**Projet final — Certification Jedha | Henintsoa**

$$Qté\_à\_commander = \max(0,\; Prédiction_{30j} + Stock\_sécurité - Stock\_actuel)$$

**Fichiers du pipeline réel** (produits par `export_stock_suppliers.py`) :
- `exports/sales_history_clean.csv` — historique de ventes (grille journalière)
- `exports/products_stock.csv` — stock disponible par produit (`qty_available`)
- `exports/suppliers.csv` — délais fournisseurs (`delay_days`, `is_preferred`)


features temporelles + lags → rolling + trends → cible → fenêtre contexte + retrait B2B → modèle.
Les features se calculent sur la série **dense** ; le nettoyage (contexte, B2B) ne fait que retirer des lignes ensuite.

In [1]:
# ============ Imports ============
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit

from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42

## 1. Chargement des données



In [2]:
# ========================
EXPORTS = "exports"
df        = pd.read_csv(f"{EXPORTS}/sales_history_clean.csv", encoding="utf-8-sig")
df["sale_date"] = pd.to_datetime(df["sale_date"])
stock     = pd.read_csv(f"{EXPORTS}/products_stock.csv", encoding="utf-8-sig")
suppliers = pd.read_csv(f"{EXPORTS}/suppliers.csv", encoding="utf-8-sig")

## 2. Filtres de périmètre

Règles métier : **ventes depuis 2023** et **hors catégorie Starlink** (gérée à part).
Le retrait **B2B**

In [3]:
avant = len(df)
df = df[df["sale_date"] >= "2023-01-01"]
df = df[~df["category"].str.contains("starlink", case=False, na=False)]
print(f"Filtres périmètre : {avant:,} → {len(df):,} lignes "
      f"({df['product_id'].nunique()} produits)")

Filtres périmètre : 4,957,672 → 4,957,672 lignes (3736 produits)


## 3. EDA — l'essentiel



In [4]:
# Tendance & saisonnalité globales
ventes_hebdo = df.set_index("sale_date").resample("W")["qty_sold"].sum().reset_index()
px.line(ventes_hebdo, x="sale_date", y="qty_sold",
        title="Ventes hebdomadaires — tendance et pics saisonniers").show()

In [5]:
# Autocorrélation → justifie lags et rolling
serie = df.groupby("sale_date")["qty_sold"].sum()
autocorr = [serie.autocorr(l) for l in range(1, 31)]
px.bar(x=list(range(1, 31)), y=autocorr,
       title="Autocorrélation (1–30 j) — les lags seront des features fortes").show()

## 4. Feature Engineering — étapes 01 et 02 du pipeline

### 4.1 Features temporelles (15) — dont jours fériés de Madagascar

In [6]:
# Jours fériés fixes (Madagascar + universels) — comme features_temporal_lags.py
FIXED_HOLIDAYS = [(1,1), (3,8), (3,29), (5,1), (6,26), (8,15), (11,1), (12,25)]

def add_temporal_features(df):
    df = df.copy()
    dt = df["sale_date"].dt
    df["day_of_week"]  = dt.dayofweek
    df["day_of_month"] = dt.day
    df["week_of_year"] = dt.isocalendar().week.astype(int)
    df["month"]        = dt.month
    df["quarter"]      = dt.quarter
    df["year"]         = dt.year
    df["is_weekend"]     = (df["day_of_week"] >= 5).astype(int)
    df["is_month_start"] = (df["day_of_month"] <= 5).astype(int)
    df["is_month_end"]   = (df["day_of_month"] >= 25).astype(int)
    df["is_holiday"] = 0
    for m, j in FIXED_HOLIDAYS:
        df.loc[(dt.month == m) & (dt.day == j), "is_holiday"] = 1
    holiday_dates = set(df.loc[df["is_holiday"] == 1, "sale_date"].dt.date)
    df["is_pre_holiday"] = df["sale_date"].apply(
        lambda x: 1 if (x.date() + timedelta(days=1)) in holiday_dates else 0)
    df["is_christmas_period"]   = ((dt.month == 12) & (dt.day >= 15)).astype(int)
    df["is_summer"]             = dt.month.isin([6, 7, 8]).astype(int)
    df["is_back_to_school"]     = ((dt.month == 9) & (dt.day <= 15)).astype(int)
    df["is_black_friday_period"]= ((dt.month == 11) & (dt.day >= 20)).astype(int)
    return df

df = df.sort_values(["product_id", "sale_date"]).reset_index(drop=True) 
df = add_temporal_features(df)

### 4.2 Lags par produit — [1, 3, 7, 14, 21, 28]

In [7]:
LAGS = [1, 3, 7, 14, 21, 28]
for lag in LAGS:
    df[f"lag_{lag}"] = df.groupby("product_id")["qty_sold"].shift(lag)

### 4.3 Rolling mean + std (5 fenêtres) et trends — `shift(1)` avant `rolling` = anti-fuite

In [8]:
ROLLING_WINDOWS = [7, 14, 30, 60, 90]
g = df.groupby("product_id")["qty_sold"]
for w in ROLLING_WINDOWS:
    df[f"rolling_mean_{w}"] = g.transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    df[f"rolling_std_{w}"]  = g.transform(lambda x: x.shift(1).rolling(w, min_periods=1).std())

# Trends : court terme vs moyen terme (>1 croissance, <1 déclin) et moyen vs long
df["trend_7_30"]  = (df["rolling_mean_7"]  / df["rolling_mean_30"].replace(0, np.nan)) \
                    .replace([np.inf, -np.inf], 0).fillna(0)
df["trend_30_90"] = (df["rolling_mean_30"] / df["rolling_mean_90"].replace(0, np.nan)) \
                    .replace([np.inf, -np.inf], 0).fillna(0)
df["category_enc"] = df["category"].astype("category").cat.codes
print("Colonnes :", len(df.columns))

Colonnes : 40


In [9]:
df

,sale_date,product_id,product_name,category,list_price,qty_sold,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend,is_month_start,is_month_end,is_holiday,is_pre_holiday,is_christmas_period,is_summer,is_back_to_school,is_black_friday_period,lag_1,lag_3,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_30,rolling_std_30,rolling_mean_60,rolling_std_60,rolling_mean_90,rolling_std_90,trend_7_30,trend_30_90,category_enc
0,2023-01-02,7827,Bose QuietComfort QC35 Noir,All / MARCHANDISES / SON HIFI / Casques,0.0,0.0,0,2,1,1,1,2023,0,1,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,66
1,2023-01-03,7827,Bose QuietComfort QC35 Noir,All / MARCHANDISES / SON HIFI / Casques,0.0,0.0,1,3,1,1,1,2023,0,1,0,0,0,0,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,0.0,66
2,2023-01-04,7827,Bose QuietComfort QC35 Noir,All / MARCHANDISES / SON HIFI / Casques,0.0,0.0,2,4,1,1,1,2023,0,1,0,0,0,0,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66
3,2023-01-05,7827,Bose QuietComfort QC35 Noir,All / MARCHANDISES / SON HIFI / Casques,0.0,0.0,3,5,1,1,1,2023,0,1,0,0,0,0,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66
4,2023-01-06,7827,Bose QuietComfort QC35 Noir,All / MARCHANDISES / SON HIFI / Casques,0.0,0.0,4,6,1,1,1,2023,0,0,0,0,0,0,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4957667,2026-08-16,29410,Lenovo Thinkbook 16-IRL,All / MARCHANDISES / LAPTOPS / Laptops Neufs,0.0,0.0,6,16,33,8,3,2026,1,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59
4957668,2026-08-17,29410,Lenovo Thinkbook 16-IRL,All / MARCHANDISES / LAPTOPS / Laptops Neufs,0.0,0.0,0,17,34,8,3,2026,0,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59
4957669,2026-08-18,29410,Lenovo Thinkbook 16-IRL,All / MARCHANDISES / LAPTOPS / Laptops Neufs,0.0,0.0,1,18,34,8,3,2026,0,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59
4957670,2026-08-19,29410,Lenovo Thinkbook 16-IRL,All / MARCHANDISES / LAPTOPS / Laptops Neufs,0.0,0.0,2,19,34,8,3,2026,0,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59


## 5. Cible : ventes des 30 prochains jours

Calculée **avant** le nettoyage (elle a besoin des jours futurs de la série dense).

In [10]:
HORIZON = 30
df["target_30j"] = (df.groupby("product_id")["qty_sold"]
    .transform(lambda s: s[::-1].rolling(HORIZON, min_periods=HORIZON).sum()[::-1].shift(-1)))

## 6. Nettoyage — étape 03 du pipeline : fenêtre contexte + retrait B2B

**Fenêtre contexte ±7 j** : on ne garde que les lignes proches d'une vente réelle.
Les longues plages de zéros des produits dormants n'apportent rien au modèle et déséquilibrent l'apprentissage.


**Retrait B2B statistique** : un produit au profil « gros pics rares » (moyenne ≥ 5 **ou** max ≥ 15, et CV ≥ 1,5)
suit une logique de devis, pas de réassort boutique → on le retire du périmètre.

In [11]:
CONTEXT_WINDOW = 7

# Fenêtre contexte — vectorisée : garder toute ligne à ±7 j d'une vente du même produit
a_vendu = (df["qty_sold"] > 0).astype(int)
df["keep"] = (a_vendu.groupby(df["product_id"])
              .transform(lambda s: s.rolling(2*CONTEXT_WINDOW + 1, center=True, min_periods=1).max()))
avant = len(df)
df = df[df["keep"] == 1].drop(columns=["keep"]).reset_index(drop=True)
print(f"Fenêtre ±{CONTEXT_WINDOW}j : {avant:,} → {len(df):,} lignes")

Fenêtre ±7j : 4,957,672 → 421,631 lignes


In [12]:
# Détection B2B statistique (mêmes seuils que filter_b2c.py)
ventes = df[df["qty_sold"] > 0]
stats = ventes.groupby(["product_id", "product_name"]).agg(
    qty_moyenne=("qty_sold", "mean"),
    qty_max=("qty_sold", "max"),
    qty_std=("qty_sold", "std"),
).reset_index()
stats["cv"] = (stats["qty_std"] / stats["qty_moyenne"]).fillna(0)
stats["is_b2b"] = (((stats["qty_moyenne"] >= 5) | (stats["qty_max"] >= 15))
                   & (stats["cv"] >= 1.5))

b2b_ids = stats.loc[stats["is_b2b"], "product_id"].tolist()
print(f"Produits B2B détectés : {len(b2b_ids)}")
print(stats.loc[stats["is_b2b"], ["product_name", "qty_moyenne", "qty_max", "cv"]]
      .head(10).to_string(index=False))

nb_avant = df["product_id"].nunique()
df = df[~df["product_id"].isin(b2b_ids)].reset_index(drop=True)
print(f"Retrait B2B : {nb_avant} → {df['product_id'].nunique()} produits B2C")

Produits B2B détectés : 27
                                                                  product_name  qty_moyenne  qty_max       cv
Adaptateur Ethernet Satellite Internet Starlink V2 pour parabole rectangulaire     2.797203     52.0 1.821200
                                                Starlink Standard Actuated Kit    23.139535    219.0 1.735698
                                    EA Sport FC 25 Standard Nintendo Switch VF     1.804878     20.0 1.699689
                                              Rouleau thermique 80x80x12mm 48g     2.983871     40.0 1.711543
                                                Blackview Tab 16 Twilight Blue     2.500000     38.0 2.800000
                                               Blackview Tab 16 Meteorite Gray     1.771429     32.0 2.106032
                                               Blackview Tab 80 Nightfall Grey     3.636364     25.0 1.991702
                                                     Blackview Tab 16 Pro Grey     3.000000  

## 7. Split temporel + XGBoost (objectif Poisson) + Optuna

**Fonction objectif `count:poisson`** — la cible est un **comptage** (masse de petites valeurs
et de zéros + quelques pics). L'objectif quadratique par défaut sur-prédit la masse des petits
produits pour limiter ses erreurs sur les pics (constaté : biais +50 %, concentré à +96 % sur
le bottom 80 % → R² négatif). L'objectif Poisson, dédié aux comptages, ramène le biais près
de zéro, fait basculer le R² en positif et réduit le MAE d'environ 25 %.

`year` est volontairement exclue des features du modèle : en production, l'année à prédire
n'a jamais été vue à l'entraînement — XGBoost ne sait pas extrapoler une valeur inconnue.

In [13]:
FEATURES = (
    [f"lag_{l}" for l in LAGS]
    + [f"rolling_mean_{w}" for w in ROLLING_WINDOWS]
    + [f"rolling_std_{w}" for w in ROLLING_WINDOWS]
    + ["trend_7_30", "trend_30_90",
       "day_of_week", "day_of_month", "week_of_year", "month", "quarter",
       "is_weekend", "is_month_start", "is_month_end",
       "is_holiday", "is_pre_holiday", "is_christmas_period",
       "is_summer", "is_back_to_school", "is_black_friday_period",
       "list_price", "category_enc"]
)
data = df.dropna(subset=FEATURES + ["target_30j"]).reset_index(drop=True)

date_coupure = data["sale_date"].max() - pd.Timedelta(days=90)
train = data[data["sale_date"] <= date_coupure]
test  = data[data["sale_date"] >  date_coupure]
X_train, y_train = train[FEATURES], train["target_30j"]
X_test,  y_test  = test[FEATURES],  test["target_30j"]
print(f"{len(FEATURES)} features | Train {len(train):,} → {date_coupure.date()} | Test {len(test):,}")

34 features | Train 372,745 → 2026-04-22 | Test 21,393


In [14]:
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

def accuracy_pm(y_true, y_pred, tol=3):
    return np.mean(np.abs(y_true - y_pred) <= tol) * 100

In [15]:
# =====  =====
def objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
        "max_depth":        trial.suggest_int("max_depth", 3, 9),      
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "random_state": RANDOM_STATE, "n_jobs": -1, "tree_method": "hist",
       
    }
    tscv = TimeSeriesSplit(n_splits=3)
    Xt, yt = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
    scores = []
    for tr, va in tscv.split(Xt):
        m = xgb.XGBRegressor(**params)
        m.fit(Xt.iloc[tr], yt.iloc[tr], verbose=False)
        scores.append(wmape(yt.iloc[va].values, m.predict(Xt.iloc[va])))
    return np.mean(scores)

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=120)          
print(study.best_params)    

{'n_estimators': 479, 'max_depth': 4, 'learning_rate': 0.013838028839186988, 'subsample': 0.7414123404000771, 'colsample_bytree': 0.6769195197453994, 'reg_alpha': 0.0010173196641310668, 'reg_lambda': 0.012796558546156345}


## 8. Évaluation

In [16]:
# ============================================================
# CONFIGURATION FINALE
# (Optuna 120 trials, seed 42 ; objectif Poisson pour le comptage)
# ============================================================
best_params = {
    "n_estimators": 479,
    "max_depth": 4,
    "learning_rate": 0.013838028839186988,
    "subsample": 0.7414123404000771,
    "colsample_bytree": 0.6769195197453994,
    "reg_alpha": 0.0010173196641310668,
    "reg_lambda": 0.012796558546156345,
    "objective": "count:poisson",
    "random_state": 42, "n_jobs": -1, "tree_method": "hist",
}
model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train, verbose=False)
y_pred = np.clip(model.predict(X_test), 0, None)

In [17]:
from sklearn.metrics import mean_absolute_error

# ===== Les 2 métriques présentées + la tolérance opérationnelle =====
print(f"MAE                  : {mean_absolute_error(y_test, y_pred):.2f} articles (erreur moyenne par produit / 30 j)")
print(f"WMAPE                : {wmape(y_test.values, y_pred):.1f} %  (= somme des erreurs / somme des ventes réelles)")
print(f"Tolérance ±3 articles: {accuracy_pm(y_test.values, y_pred):.1f} %  (prédictions opérationnellement justes)")

# 
biais = (y_pred.sum() / y_test.sum() - 1) * 100
print(f"[contrôle] biais     : {biais:+.1f} %  (>0 = tendance surstock, <0 = tendance rupture)")

MAE                  : 3.03 articles (erreur moyenne par produit / 30 j)
WMAPE                : 92.5 %  (= somme des erreurs / somme des ventes réelles)
Tolérance ±3 articles: 86.5 %  (prédictions opérationnellement justes)
[contrôle] biais     : +20.6 %  (>0 = tendance surstock, <0 = tendance rupture)


In [18]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values().tail(15)
px.bar(imp, orientation="h", title="Top 15 features (XGBoost)").show()

In [19]:
mask_ps5 = test["product_name"] == "Console PS5"
if mask_ps5.sum() > 0:
    err = np.abs(y_test[mask_ps5].values - y_pred[mask_ps5])
    print(f"PS5 — MAE sur l'horizon 30 j : {err.mean():.2f} articles")
    comp = test.loc[mask_ps5, ["sale_date"]].assign(
        reel=y_test[mask_ps5].values, prediction=y_pred[mask_ps5])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=comp["sale_date"], y=comp["reel"], name="Réel"))
    fig.add_trace(go.Scatter(x=comp["sale_date"], y=comp["prediction"], name="Prédiction"))
    fig.update_layout(title="PS5 — demande 30 j : réel vs prédit").show()

## 9. Réassort — avec les vrais délais fournisseurs

Le stock de sécurité utilise le **délai du fournisseur préféré** de chaque produit
(`suppliers.csv`, colonne `delay_days`) :
$$SS = Z \times \sigma_{30} \times \sqrt{délai}$$

In [20]:
from datetime import datetime, timedelta as td

Z = 1.65
DELAI_DEFAUT = 5

dernier = (data.sort_values("sale_date").groupby("product_id").tail(1)
               .set_index("product_id"))

# Fournisseur préféré : délai, nom, prix d'achat
pref = (suppliers[suppliers["is_preferred"]]
        .sort_values("delay_days").groupby("product_id").first())

r = pd.DataFrame(index=dernier.index)
r["product_name"]     = dernier["product_name"]
r["category_id"]      = dernier["category"]          # voir note catég. ci-dessous
r["ventes_prevues_30j"] = np.clip(model.predict(dernier[FEATURES]), 0, None).round(1)
r["avg_daily_demand"] = (r["ventes_prevues_30j"] / 30).round(2)
r["qty_available"]    = r.index.map(stock.set_index("product_id")["qty_available"]).fillna(0)
r["delay_days"]       = r.index.map(pref["delay_days"]).fillna(DELAI_DEFAUT).astype(int)
r["supplier_name"]    = r.index.map(pref["supplier_name"]).fillna("Non renseigné")
r["prix_achat"]       = r.index.map(pref["supplier_price"]).fillna(0)
r["prix_vente"]       = dernier["list_price"]
r["marge_pct"]        = np.where(r["prix_vente"] > 0,
                                 ((r["prix_vente"] - r["prix_achat"]) / r["prix_vente"] * 100),
                                 0).round(1)

# Cycle de vie & obsolescence (dérivés des trends)
t730, t3090 = dernier["trend_7_30"], dernier["trend_30_90"]
r["cycle_status"] = np.select(
    [t730 >= 1.15, (t730 <= 0.85) & (t730 > 0), t730 == 0],
    ["Croissance", "Déclin", "Dormant"], default="Maturité")
r["obsolescence_score"] = np.clip(1 - t3090, 0, 1).round(3)   # 0 = en forme, 1 = obsolète

# Stock de sécurité, quantité à commander, couverture
r["safety_stock"] = (Z * dernier["rolling_std_30"].fillna(0) * np.sqrt(r["delay_days"])).round(1)
r["qty_to_order"] = np.maximum(0, r["ventes_prevues_30j"] + r["safety_stock"]
                                  - r["qty_available"]).round(0).astype(int)
r["coverage_days"] = np.where(r["avg_daily_demand"] > 0,
                              r["qty_available"] / r["avg_daily_demand"], 999).round(1)
r["estimated_cost"] = (r["qty_to_order"] * r["prix_achat"]).round(0)

# Dates : livraison idéale = épuisement du stock ; commander avant = livraison − délai
auj = datetime.now().date()
r["ideal_delivery"] = [(auj + td(days=min(c, 365))).isoformat() for c in r["coverage_days"]]
r["order_by_date"]  = [(auj + td(days=max(min(c, 365) - d, 0))).isoformat()
                       for c, d in zip(r["coverage_days"], r["delay_days"])]

# Alertes (libellés texte, comme l'ancienne version)
def alerte(row):
    if row["ventes_prevues_30j"] > 0 and row["qty_available"] <= 0: return "Rupture"
    if row["coverage_days"] < row["delay_days"]:                    return "Rupture imminente"
    if row["coverage_days"] < row["delay_days"] + 7:                return "Stock bas"
    if row["coverage_days"] < 30:                                   return "À surveiller"
    return "OK"
r["alert"] = r.apply(alerte, axis=1)
prio = {"Rupture": 0, "Rupture imminente": 1, "Stock bas": 2, "À surveiller": 3, "OK": 4}
tableau = r.sort_values(["alert", "qty_to_order"],
                        key=lambda s: s.map(prio) if s.name == "alert" else s,
                        ascending=[True, False])
print(tableau["alert"].value_counts().to_string())
tableau.head(15)

alert
OK              3067
Rupture          495
À surveiller      76


,product_name,category_id,ventes_prevues_30j,avg_daily_demand,qty_available,delay_days,supplier_name,prix_achat,prix_vente,marge_pct,cycle_status,obsolescence_score,safety_stock,qty_to_order,coverage_days,estimated_cost,ideal_delivery,order_by_date,alert
product_id,,,,,,,,,,,,,,,,,,,
29327,Blackview Airbuds 16 Noir,All / MARCHANDISES / SON HIFI / Ecouteurs,17.1,0.57,0.0,0,NEXTHOPE,40783.30,1.0,-4078230.0,Déclin,0.000,0.0,17,0.0,693316.0,2026-09-05,2026-09-05,Rupture
19313,Lenovo Thinkpad T490s,All / MARCHANDISES / LAPTOPS / Laptops Recondi...,9.3,0.31,0.0,0,MASS' IN,1390727.20,2200000.0,36.8,Déclin,0.000,0.0,9,0.0,12516545.0,2026-09-05,2026-09-05,Rupture
19503,Lenovo Thinkpad T490,All / MARCHANDISES / LAPTOPS / Laptops Recondi...,8.9,0.30,0.0,0,NEXTHOPE,737900.00,1.0,-73789900.0,Croissance,0.000,0.0,9,0.0,6641100.0,2026-09-05,2026-09-05,Rupture
19210,Lenovo Thinkpad T490s,All / MARCHANDISES / LAPTOPS / Laptops Recondi...,8.9,0.30,0.0,0,MASS' IN,1658270.10,1.0,-165826910.0,Déclin,0.000,0.0,9,0.0,14924431.0,2026-09-05,2026-09-05,Rupture
20438,Audio Technica - AT2040USB,All / MARCHANDISES / HOMESTUDIO / MICROPHONE D...,8.4,0.28,0.0,0,NEXTHOPE,585756.00,1.0,-58575500.0,Croissance,0.000,0.0,8,0.0,4686048.0,2026-09-05,2026-09-05,Rupture
29260,Blackview Wave 10 Blue,All / MARCHANDISES / SMARTPHONES / Smartphones...,7.6,0.25,0.0,0,NEXTHOPE,414868.65,1.0,-41486765.0,Déclin,0.000,0.0,8,0.0,3318949.0,2026-09-05,2026-09-05,Rupture
19517,Lenovo Thinkpad T490,All / MARCHANDISES / LAPTOPS / Laptops Recondi...,7.0,0.23,0.0,0,NEXTHOPE,737900.00,1.0,-73789900.0,Croissance,0.000,0.0,7,0.0,5165300.0,2026-09-05,2026-09-05,Rupture
20511,Lenovo Thinkpad T490s,All / MARCHANDISES / LAPTOPS / Laptops Recondi...,7.1,0.24,0.0,0,NEXTHOPE,1166000.00,2200000.0,47.0,Déclin,0.000,0.0,7,0.0,8162000.0,2026-09-05,2026-09-05,Rupture
29249,Blackview Tab 60 Pro Set Blue,All / MARCHANDISES / TABLETTE ET TELEPHONIE / ...,7.0,0.23,0.0,0,NEXTHOPE,521115.93,0.0,0.0,Croissance,0.000,0.0,7,0.0,3647812.0,2026-09-05,2026-09-05,Rupture


In [21]:
export = tableau.reset_index()[
    ["product_id", "product_name", "category_id", "cycle_status", "obsolescence_score",
     "ventes_prevues_30j", "avg_daily_demand", "qty_available", "coverage_days",
     "safety_stock", "qty_to_order", "prix_vente", "prix_achat", "marge_pct",
     "supplier_name", "delay_days", "estimated_cost", "ideal_delivery",
     "order_by_date", "alert"]].copy()
export.to_csv("reassort.csv", index=False, encoding="utf-8-sig")
print(f" reassort.csv — {len(export)} produits")

 reassort.csv — 3638 produits


In [ ]:
# ============================================================
# Export du modèle → model/model.pkl
# ============================================================
import pickle
from pathlib import Path
from datetime import datetime

Path("model").mkdir(exist_ok=True)

artefact = {
    "model": model,                          # le XGBRegressor entraîné
    "features": FEATURES,                  
    "params": best_params,               
    "horizon_jours": HORIZON,                
    "metriques_test": {
        "mae": 3.03, "wmape_pct": 92.5, "tolerance_pm3_pct": 86.6
    },
    "date_entrainement": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "version_xgboost": xgb.__version__,     
}

with open("model/model.pkl", "wb") as f:
    pickle.dump(artefact, f)

taille = Path("model/model.pkl").stat().st_size / 1e6
print(f"✅ model/model.pkl ({taille:.1f} Mo) — xgboost {xgb.__version__}")

✅ model/model.pkl (0.8 Mo) — xgboost 3.2.0


---
##  Récapitulatif

1. **Périmètre** : depuis 2023, hors Starlink ; B2B retiré **statistiquement** (profil gros pics rares) car non marqué en base
2. **Features** (fidèles au pipeline) : 6 lags + rolling mean/std ×5 fenêtres + 2 trends + 15 temporelles (jours fériés Madagascar) — `shift` avant `rolling`, `year` exclue du modèle
3. **Nettoyage après features** : fenêtre contexte ±7 j (zéros isolés) — l'ordre 01→02→03 garantit des lags calculés sur série dense
4. **Modèle** : XGBoost + Optuna, split temporel, WMAPE + MAE ±3
5. **Réassort** : `max(0, prédiction + Z·σ₃₀·√délai − stock)` avec les **vrais délais fournisseurs** → `reassort.csv`

**Améliorations** : stock historique (stock_move) pour un vrai indicateur de rupture, segmentation avant modélisation, LightGBM challenger, monitoring de dérive.